In [ ]:
!pip install librosa imbalanced-learn scikit-learn


In [ ]:
import os, glob
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader


In [ ]:
def get_paths_labels(folder_path, label):
    files = glob.glob(os.path.join(folder_path, "*.wav"))
    return [(f, label) for f in files]

def extract_lfcc(path, sr=16000, n_mfcc=20):
    y, _ = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # Proxy for LFCC
    return np.mean(mfcc.T, axis=0)  # mean pooling (1D vector)


In [ ]:
class LFCCDNN_Dataset(Dataset):
    def __init__(self, data_pairs):
        self.X, self.y = [], []
        for path, label in data_pairs:
            try:
                features = extract_lfcc(path)
                self.X.append(torch.tensor(features, dtype=torch.float32))
                self.y.append(label)
            except Exception as e:
                print(f"Error loading {path}: {e}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], torch.tensor(self.y[idx])


In [ ]:
class DNNModel(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64):
        super(DNNModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
train_real = get_paths_labels("/kaggle/input/scenefake/train/real", 0)
train_fake = get_paths_labels("/kaggle/input/scenefake/train/fake", 1)
test_real  = get_paths_labels("/kaggle/input/scenefake/eval/real", 0)
test_fake  = get_paths_labels("/kaggle/input/scenefake/eval/fake", 1)

train_data = train_real + train_fake
test_data = test_real + test_fake

def extract_features_labels(data_pairs):
    X, y = [], []
    for path, label in data_pairs:
        try:
            feat = extract_lfcc(path)
            X.append(feat)
            y.append(label)
        except:
            continue
    return np.array(X), np.array(y)

X_train, y_train = extract_features_labels(train_data)

# SMOTE for class balance
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

# Convert to tensors
X_tensor = torch.tensor(X_train_bal, dtype=torch.float32)
y_tensor = torch.tensor(y_train_bal, dtype=torch.long)
train_dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)

# ✅ Define test_dataset properly
test_dataset = LFCCDNN_Dataset(test_data)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DNNModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(40):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")


In [ ]:
model.eval()
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
print(f"Equal Error Rate (EER): {eer:.4f}")
